[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_05/SFM_ch5_evt_var/SFM_ch5_evt_var.ipynb)

# SFM_ch5_evt_var

VaR 1%, ES 2.5% and VaR 0.1% of daily losses by extreme value theory (peaks over threshold, u at the 90% quantile), by historical simulation, by the Normal distribution and by the Hill quantile, for the S&P 500, DAX, BET, Bitcoin and four BVB stocks; an out-of-sample check: VaR estimated until 2019 and the number of exceedances in 2020-2026. Based on the Quantlet var_pot.

*Statistics of Financial Markets (SFM), Chapter 5: heavy tails and extreme value theory. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/SFM/Chapter_5'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import sfm_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the SFM repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years

## Analysis

In [ ]:
from scipy import optimize

START = {'sp500': '1990-01-01', 'dax': '1990-01-01', 'bet': '1997-01-01', 'btc': None, 'snp': '2010-01-01', 'brd': '2010-01-01', 'tlv': '2010-01-01', 'tgn': '2010-01-01'}

ASSETS = ['sp500', 'dax', 'bet', 'btc']

STOCKS = ['snp', 'brd', 'tlv', 'tgn']

BAD_DAYS = {'tlv': ['2016-05-30', '2016-05-31']}

COLORS = {'sp500': '#1A3A6E', 'dax': '#17A2B8', 'bet': '#CD0000', 'btc': '#B5853F', 'snp': '#2E7D32', 'brd': '#8E44AD', 'tlv': '#E67E22', 'tgn': '#DC3545'}

MODEL_COL = {'Data': '#B5853F', 'Normal': '#CD0000', 'EVT': '#1A3A6E', 'Historical': '#2E7D32', 'Hill': '#8E44AD'}

SEED = 2026

HILL_FRAC = 0.025

POT_Q = 0.9

SPLIT = '2019-12-31'

BLOCK = 20

def returns(k):
    """Daily log returns in % on the series' own calendar (S&P 500 and DAX since 1990, BET since its launch in
    1997, Bitcoin since 2014, BVB stocks since 2010); known data errors removed."""
    r = log_returns(k, START.get(k))
    if k in BAD_DAYS:
        r = r.drop(pd.to_datetime(BAD_DAYS[k]), errors='ignore')
    return r


def losses(k):
    """Daily losses L_t = -r_t in % (a positive number is a loss)."""
    return -returns(k)


def hill(x, k):
    """Hill (1975): alpha_hat(k) = [ (1/k) sum_{i=1..k} ln(X_(i) / X_(k+1)) ]^(-1), X_(1) >= X_(2) >= ..."""
    x = np.sort(np.asarray(x, float))[::-1]
    return 1.0 / np.mean(np.log(x[:k] / x[k]))


def hill_quantile(x, k, p):
    """Hill (Weissman, 1978) quantile: x_p = X_(k+1) (k / (n p))^(1/alpha_hat), the loss exceeded with probability p."""
    x = np.sort(np.asarray(x, float))[::-1]
    return x[k] * (k / (len(x) * p)) ** (1 / hill(x, k))


def ls_tail_index(x, k):
    """Least-squares tail index (as in SFElshill): slope of ln(i/n) on ln X_(i), i = 1..k, is -alpha."""
    n = len(x)
    x = np.sort(np.asarray(x, float))[::-1][:k]
    return -np.polyfit(np.log(x), np.log(np.arange(1, k + 1) / n), 1)[0]


def block_indices(n, block, rng):
    """Indices of one moving-block bootstrap sample (blocks of `block` consecutive days)."""
    nb = int(np.ceil(n / block))
    s = rng.integers(0, n - block + 1, nb)
    return (s[:, None] + np.arange(block)[None, :]).ravel()[:n]


def hill_inference(x, frac=HILL_FRAC, B=500, block=BLOCK, seed=SEED):
    """Hill alpha_hat at k = frac n with the i.i.d. standard error alpha/sqrt(k), the 95% interval
    alpha (1 +/- 1.96/sqrt(k)) and the moving-block bootstrap standard error (blocks of 20 days)."""
    x = np.asarray(x, float)
    n = len(x)
    k = int(frac * n)
    a = hill(x, k)
    rng = np.random.default_rng(seed)
    bs = np.array([hill(x[block_indices(n, block, rng)], k) for _ in range(B)])
    return {'n': n, 'k': k, 'u': float(np.sort(x)[::-1][k]), 'alpha': a, 'se_iid': a / np.sqrt(k),
            'lo': a * (1 - 1.96 / np.sqrt(k)), 'hi': a * (1 + 1.96 / np.sqrt(k)), 'se_block': float(np.std(bs, ddof=1)),
            'alpha_1': hill(x, int(0.01 * n)), 'alpha_5': hill(x, int(0.05 * n)), 'ls': ls_tail_index(x, k)}


def num_hessian(f, p, h=1e-4):
    """Numerical Hessian of f at p (central differences)."""
    p = np.asarray(p, float)
    k = len(p)
    H = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            e_i, e_j = np.eye(k)[i] * h, np.eye(k)[j] * h
            H[i, j] = (f(p + e_i + e_j) - f(p + e_i - e_j) - f(p - e_i + e_j) + f(p - e_i - e_j)) / (4 * h * h)
    return H


def gpd_sf(y, xi, beta):
    """GPD survival function P(Y > y) = (1 + xi y / beta)^(-1/xi) for the excess Y = X - u (xi = 0: exponential)."""
    return stats.genpareto.sf(y, xi, 0, beta)


def fit_gpd(y):
    """Maximum likelihood GPD fit (xi, beta) of excesses y > 0, standard errors from the numerical Hessian."""
    y = np.asarray(y, float)
    c, _, b = stats.genpareto.fit(y, floc=0)
    nll = lambda p: -np.sum(stats.genpareto.logpdf(y, p[0], 0, p[1])) if p[1] > 0 else np.inf
    res = optimize.minimize(nll, [c, b], method='Nelder-Mead', options={'xatol': 1e-9, 'fatol': 1e-9, 'maxiter': 5000})
    xi, beta = res.x
    se = np.sqrt(np.diag(np.linalg.inv(num_hessian(nll, res.x))))
    return {'xi': xi, 'beta': beta, 'se_xi': se[0], 'se_beta': se[1], 'nll': res.fun, 'n_u': len(y)}


def pot(x, q=POT_Q):
    """GPD fit to the losses above the threshold u = q-quantile of the losses x."""
    x = np.asarray(x, float)
    u = float(np.quantile(x, q))
    f = fit_gpd(x[x > u] - u)
    f.update({'u': u, 'n': len(x), 'q': q})
    return f


def pot_var(f, p):
    """VaR_p = u + (beta/xi) [ (n p / N_u)^(-xi) - 1 ], the loss exceeded with probability p."""
    return f['u'] + f['beta'] / f['xi'] * ((f['n'] * p / f['n_u']) ** (-f['xi']) - 1)


def pot_es(f, p):
    """ES_p = VaR_p / (1 - xi) + (beta - xi u) / (1 - xi), the mean loss beyond VaR_p (xi < 1)."""
    return pot_var(f, p) / (1 - f['xi']) + (f['beta'] - f['xi'] * f['u']) / (1 - f['xi'])


def pot_tail_prob(f, x):
    """P(L > x) = (N_u / n) (1 + xi (x - u) / beta)^(-1/xi) for x > u."""
    return f['n_u'] / f['n'] * (1 + f['xi'] * (np.asarray(x, float) - f['u']) / f['beta']) ** (-1 / f['xi'])


def normal_var(x, p):
    """Normal VaR_p of losses x: mu_L + sigma_L z_(1-p)."""
    return float(np.mean(x) + np.std(x, ddof=1) * stats.norm.isf(p))


def normal_es(x, p):
    """Normal ES_p of losses x: mu_L + sigma_L phi(z_p) / p."""
    return float(np.mean(x) + np.std(x, ddof=1) * stats.norm.pdf(stats.norm.isf(p)) / p)


def hist_var(x, p):
    """Historical VaR_p: the empirical (1 - p) quantile of the losses."""
    return float(np.quantile(x, 1 - p))


def hist_es(x, p):
    """Historical ES_p: the mean of the losses at or above the historical VaR_p."""
    x = np.asarray(x, float)
    return float(x[x >= hist_var(x, p)].mean())


def risk_table(names=ASSETS):
    """VaR 1%, ES 2.5% and VaR 0.1% of daily losses: EVT (POT), historical simulation, Normal; Hill quantile."""
    out = {}
    for k in names:
        x = losses(k).values
        f = pot(x)
        kk = int(HILL_FRAC * len(x))
        out[k] = {'pot': f, 'n': len(x)}
        for lab, p in [('var1', 0.01), ('var01', 0.001)]:
            out[k][lab] = {'EVT': pot_var(f, p), 'Historical': hist_var(x, p), 'Normal': normal_var(x, p),
                           'Hill': hill_quantile(x, kk, p)}
        out[k]['es25'] = {'EVT': pot_es(f, 0.025), 'Historical': hist_es(x, 0.025), 'Normal': normal_es(x, 0.025)}
        out[k]['max'] = float(x.max())
    return out


def out_of_sample(names=ASSETS, split=SPLIT):
    """Estimate VaR 1% and VaR 0.1% on the losses until `split` (EVT, historical, Normal); count the days after
    `split` with a loss above each VaR (expected rates: 1% and 0.1%)."""
    out = {}
    for k in names:
        L = losses(k)
        a, b = L.loc[:split].values, L.loc[split:].iloc[1:].values
        f = pot(a)
        row = {'n_est': len(a), 'n_test': len(b), 'first_test': L.loc[split:].index[1].date().isoformat()}
        for lab, p in [('var1', 0.01), ('var01', 0.001)]:
            v = {'EVT': pot_var(f, p), 'Historical': hist_var(a, p), 'Normal': normal_var(a, p)}
            row[lab] = {m: {'var': x, 'exc': int((b > x).sum())} for m, x in v.items()}
            row[lab]['expected'] = p * len(b)
        out[k] = row
    return out


def fig_oos(k='sp500', split=SPLIT, save=True):
    """Daily losses after the estimation period with the VaR 1% of the three methods (estimated until 2019)."""
    L = losses(k)
    a, b = L.loc[:split].values, L.loc[split:].iloc[1:]
    f = pot(a)
    fig, ax = plt.subplots(figsize=(10, 4.0))
    ax.plot(b.index, b.values, color=COLORS[k], lw=0.6, label=f'{LABELS[k]} daily loss (%)')
    for m, v, ls in [('Normal', normal_var(a, 0.01), '--'), ('Historical', hist_var(a, 0.01), ':'), ('EVT', pot_var(f, 0.01), '-')]:
        ax.axhline(v, color=MODEL_COL[m], lw=1.3, ls=ls, label=f'VaR 1%, {m}: {v:.2f}%')
    v01 = pot_var(f, 0.001)
    ax.axhline(v01, color=MODEL_COL['EVT'], lw=1.0, ls='-.', label=f'VaR 0.1%, EVT: {v01:.2f}%')
    ax.set_ylabel('Daily loss (%)')
    st.legend_outside_bottom(ax, ncol=3, y=-0.12)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch5_oos')

## Run

In [ ]:
rt = risk_table(ASSETS + STOCKS)
print(pd.DataFrame({k: {f'{l} {m}': v[l][m] for l in ['var1', 'es25', 'var01'] for m in v[l]} for k, v in rt.items()}).T.round(2))
oos = out_of_sample()
print(pd.DataFrame({k: {f'{l} {m}': o[l][m]['exc'] for l in ['var1', 'var01'] for m in ['EVT', 'Historical', 'Normal']} for k, o in oos.items()}).T)
fig_oos()

![sfm_ch5_oos](./sfm_ch5_oos.png)

Output: `sfm_ch5_oos.pdf`